![picture](https://drive.google.com/uc?export=view&id=1eCsjNAtjXuXfqBLxeEnsBpOikUO06msr)

# **NLP701@MBZUAI Fall 2025 - Lab 10**



## **Learning Outcomes**
- Critically analyze, evaluate, and improve the performance of RNN and LSTM.
- Gain proficiency in implementing neural networks.

## **Learning Activities**
- Implement an NER tagger using RNNs.
- Improving the model by adding character level information or other methods.


# **RNNs for Sequence Labeling**
In this lab exercise, we build an NER tagger for Arabic using **RNNs**.
We use the same dataset as assignment 1, i.e., ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).

## **Data Preparation**

In [1]:
!wget "https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip"
!unzip ANERcorp-CamelLabSplits.zip



7[Files: 0  Bytes: 0  [0 B/s] Re]87[https://camel.abudhabi.nyu.edu]87Saving 'ANERcorp-CamelLabSplits.zip'
87ANERcorp-CamelLabSpl 100% [=============================>]  904.70K    --.-KB/s87HTTP response 200 Accept-Ranges: bytes [https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip]
87ANERcorp-CamelLabSpl 100% [=============================>]  904.70K    --.-KB/s87[Files: 1  Bytes: 904.70K [1.12]8Archive:  ANERcorp-CamelLabSplits.zip
   creating: ANERcorp-CamelLabSplits/
  inflating: __MACOSX/._ANERcorp-CamelLabSplits  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_Benajiba.txt  
  inflating: __MACOSX/ANERcorp-CamelLabSplits/._ANERCorp_Benajiba.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt  
  inflating: __MACOSX/ANERcorp-CamelLabSplits/._ANERCorp_CamelLab_train.txt  
  inflating: ANERcorp-CamelLabSplits/README.txt  
  inflating: __MACOSX/ANERcorp-CamelLabSplits/._README.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_

In [2]:
from sklearn.model_selection import train_test_split

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

In [3]:
# check the dataset
print('Size of train_sents: %d'%len(train_sents))
print('Size of dev_sents: %d'%len(dev_sents))

print('The training data looks like:')
for sent in train_sents[:5]:
  print(sent)

Size of train_sents: 3575
Size of dev_sents: 398
The training data looks like:
[('وقال', 'O'), ('إن', 'O'), ('العملية', 'O'), ('استهدفت', 'O'), ('"', 'O'), ('بنى', 'O'), ('تحتية', 'O'), ('تستعمل', 'O'), ('لتخزين', 'O'), ('أسلحة', 'O'), ('عائدة', 'O'), ('للجهاد', 'O'), ('الإسلامي', 'O'), ('"', 'O'), ('،', 'O'), ('واوضح', 'O'), ('إن', 'O'), ('إسرائيل', 'B-LOC'), ('أخطرت', 'O'), ('سكان', 'O'), ('المنطقة', 'O'), ('بوقوع', 'O'), ('الغارة', 'O'), ('.', 'O')]
[('وهذه', 'O'), ('هي', 'O'), ('المباراة', 'O'), ('الرسمية', 'O'), ('الأولى', 'O'), ('لمنتخب', 'O'), ('إنجلترا', 'B-LOC'), ('تحت', 'O'), ('قيادة', 'O'), ('مدربه', 'O'), ('ستيف', 'B-PER'), ('مكلارين', 'I-PER'), ('الذي', 'O'), ('خلف', 'O'), ('السويدي', 'O'), ('غوران', 'B-PER'), ('إريكسون', 'I-PER'), ('بعد', 'O'), ('نهائيات', 'O'), ('المونديال', 'B-MISC'), ('.', 'O')]
[('في', 'O'), ('عيد', 'O'), ('الملكة', 'O'), ('السماوية', 'O'), ('"', 'O'), ('تيني', 'B-PER'), ('هاو', 'I-PER'), ('"', 'O'), ('يسود', 'O'), ('إحساس', 'O'), ('بالروحانية', 'O'),

## **Build Vocabulary**

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

torch.manual_seed(1)

def build_vocab(sents, idx, special_tokens=['UNK']):
    vocab = {}
    if special_tokens:
        vocab = {k: v for v, k in enumerate(special_tokens)}

    for sent in sents:
        for word in sent:
            if word[idx] not in vocab:
                vocab[word[idx]] = len(vocab)

    return vocab

def prepare_sequence(seq, to_idx):
    idxs = []
    for w in seq:
        if w in to_idx:
            idxs.append(to_idx[w])
        else:
            idxs.append(to_idx['UNK'])
    return torch.tensor(idxs, dtype=torch.long)

In [5]:
word_to_idx = build_vocab(train_sents, 0)
tag_to_idx = build_vocab(train_sents, 1, special_tokens=None)

## **Create the model**

In [6]:
class RNNTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(RNNTagger, self).__init__()
        self.hidden_dim = hidden_dim
        # Word embedding layer. This maps each word to a vector representation.
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The RNN takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        # You can utilize the multi-layer rnn by changing num_layers.
        # Also you can use the bidirectional rnn with bidirectional=True.
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers=1, bidirectional=False)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        rnn_out, _ = self.rnn(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(rnn_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

    def predict(self, sentence):
        with torch.no_grad():
            inputs = prepare_sequence(sentence, word_to_idx)
            tag_scores = self.forward(inputs)
            _, indices = torch.max(tag_scores, 1)
            tags = []
            for i in range(len(indices)):
                for key, value in tag_to_idx.items():
                    if indices[i] == value:
                        tags.append(key)
        return tags

## **Train the model**

In [7]:
def train(model, train_sents, lr=0.1, epochs=3):
    loss_function = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for epoch in range(epochs):

        for sentence in tqdm(train_sents):
            # Step 1. Remember that Pytorch accumulates gradients.
            # We need to clear them out before each instance
            optimizer.zero_grad() # or model.zero_grad()

            # Step 2. Get our inputs ready for the network, that is, turn them into
            # Tensors of word indices.
            x = [word[0] for word in sentence]
            y = [word[1] for word in sentence]
            sentence_in = prepare_sequence(x, word_to_idx)
            targets = prepare_sequence(y, tag_to_idx)

            # Step 3. Run our forward pass.
            tag_scores = model(sentence_in)

            # Step 4. Compute the loss, gradients, and update the parameters by
            #  calling optimizer.step()
            loss = loss_function(tag_scores, targets)
            loss.backward()
            optimizer.step()

        print(f'epoch: {epoch+1}, loss: {loss}')

    return model

In [8]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

RNN_model = RNNTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_idx), len(tag_to_idx))
RNN_tagger = train(RNN_model, train_sents, epochs=5)

torch.save(RNN_tagger.state_dict(), './rnn_tagger.pt')

100%|██████████| 3575/3575 [00:50<00:00, 70.86it/s] 


epoch: 1, loss: 0.1423054039478302


 61%|██████    | 2183/3575 [00:17<00:10, 127.07it/s]


KeyboardInterrupt: 

## **Evaluate the Model**

Load the model:

In [ ]:
RNN_tagger = RNNTagger(EMBEDDING_DIM,
                       HIDDEN_DIM,
                       len(word_to_idx),
                       len(tag_to_idx))
RNN_tagger.load_state_dict(torch.load('./rnn_tagger.pt'))
RNN_tagger.eval()

Evalute the model:

In [ ]:
from seqeval.metrics import f1_score

dev = [[word[0] for word in sent] for sent in dev_sents]
dev_gold = [[word[1] for word in sent] for sent in dev_sents]

dev_pred = [RNN_tagger.predict(sent) for sent in dev]
f1_score(dev_gold, dev_pred)

# **Exercise 01: LSTM**

Create the model:

In [ ]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim
        # Word embedding layer. This maps each word to a vector representation.
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        # You can utilize the multi-layer LSTM by changing num_layers.
        # Also you can use the bidirectional LSTM with bidirectional=True.
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=1, bidirectional=False)

        # The linear layer that maps from hidden state space to tag space
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        tag_scores = F.log_softmax(tag_space, dim=1)
        return tag_scores

    def predict(self, sentence):
        with torch.no_grad():
            inputs = prepare_sequence(sentence, word_to_idx)
            tag_scores = self.forward(inputs)
            _, indices = torch.max(tag_scores, 1)
            tags = []
            for i in range(len(indices)):
                for key, value in tag_to_idx.items():
                    if indices[i] == value:
                        tags.append(key)
        return tags

Train the model:

In [ ]:
EMBEDDING_DIM = 128
HIDDEN_DIM = 256

# Create and train the LSTM model
LSTM_model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_idx), len(tag_to_idx))
LSTM_tagger = train(LSTM_model, train_sents, epochs=5)

# Save the trained model
torch.save(LSTM_tagger.state_dict(), './lstm_tagger.pt')

Evalute the model:

In [ ]:
# Load the LSTM model (if needed)
# LSTM_tagger = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_idx), len(tag_to_idx))
# LSTM_tagger.load_state_dict(torch.load('./lstm_tagger.pt'))
# LSTM_tagger.eval()

# Evaluate the LSTM model on dev set
from seqeval.metrics import f1_score, classification_report

dev = [[word[0] for word in sent] for sent in dev_sents]
dev_gold = [[word[1] for word in sent] for sent in dev_sents]

# Get predictions
dev_pred = [LSTM_tagger.predict(sent) for sent in dev]

# Calculate F1 score
lstm_f1 = f1_score(dev_gold, dev_pred)
print(f'LSTM F1 Score: {lstm_f1:.4f}')

# Print detailed classification report
print('\nClassification Report:')
print(classification_report(dev_gold, dev_pred))

## **What would be a good example to show that LSTM performs better than a simple RNN?**
- No coding involved, and no need to use Arabic as an example.
- Imagine you're writing a paper, and you want to show an example that works well in LSTM compared to RNN.
- What kind of example would you present and why?

In [ ]:
# Example demonstrating LSTM superiority over simple RNN:

"""
**Example: Long-Distance Agreement in Language**

Input sentence: "The keys to the cabinet that has been sitting in the storage 
room for several years are missing."

Task: Predict the verb number agreement (singular/plural)

**Why LSTM performs better:**

1. **Long-term dependencies:** The subject "keys" (plural) is separated from the 
   verb "are" by 14 words, including a singular noun "cabinet" that could mislead 
   the model.

2. **RNN limitation:** Simple RNNs suffer from vanishing gradients over long sequences.
   By the time the RNN reaches "are", the information about "keys" being plural has
   largely faded. The RNN might incorrectly predict singular agreement based on the
   more recent noun "room" or "cabinet".

3. **LSTM advantage:** The LSTM's memory cells and gating mechanisms (forget gate,
   input gate, output gate) allow it to:
   - Retain the crucial information that "keys" is the subject
   - Ignore intervening nouns like "cabinet" and "room"
   - Correctly predict plural agreement when it reaches "are"

**Expected Results:**
- RNN: Might predict 60-70% accuracy on long-distance agreement
- LSTM: Typically achieves 85-95% accuracy on the same task

This example clearly demonstrates LSTM's ability to maintain and selectively access
information across long sequences, which is critical for NLP tasks requiring 
understanding of syntactic structure and long-range dependencies.
"""

# **Bonus Exercise: How Would You Improve the Model?**
- What can be done on top of the simple LSTM tagger to improve the performance?

In [ ]:
#[Your Code]

## **Reference**
https://pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html